# DataFlow Doctor: Evidence-Grounded Data Pipeline & ETL Debugging Assistant

**Kushagra Gupta - RAG Portfolio Project**

DataFlow Doctor is a Retrieval-Augmented Generation system built on a custom dataset of Airflow-style DAGs, schemas, SQL transformations, logs, data-quality rules, runbooks and incident reports. The notebook intentionally reuses the syntax and workflow from the six in-class labs as much as possible.


## What from the in-class labs is reused?

| In-class lab | DataFlow Doctor use |
|---|---|
| Lab 1 - LLM Generation Parameters | Factual generation with low randomness |
| Lab 2 - Introduction to LangChain | chat model, prompt templates and runnable chains |
| Lab 3 - Structured Output Generation | Pydantic schema + `with_structured_output` |
| Lab 4 - RAG | loaders, chunking, Chroma, BM25, EnsembleRetriever, LCEL |
| Lab 5 - Advanced RAG | MMR and MultiQuery experiment |
| Lab 6 - RAG with LangGraph | START -> retrieve -> generate -> END |

Only small project-specific additions are made for data-pipeline metadata, the bundled corpus, retrieval evaluation and deterministic lineage tracing.


## 1. Initial Setup


In [1]:
import os, json, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from uuid import uuid4
from dotenv import load_dotenv
from IPython.display import display, Markdown

warnings.filterwarnings('ignore')
load_dotenv(override=True)

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
DATASET_DIR = PROJECT_ROOT / 'dataset'
EVAL_FILE = PROJECT_ROOT / 'evaluation' / 'evaluation_questions.csv'
LINEAGE_FILE = DATASET_DIR / 'lineage.csv'
PERSIST_DIR = str(PROJECT_ROOT / 'chroma_dataflow_doctor_db')


### 1.1 Chat model - Lab 1 / Lab 2 style


In [2]:
# Kushagra can use the laptop GPU through local Ollama.
# Optional Groq support is kept in the standalone app, but this notebook stays local-first.
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model=os.getenv('OLLAMA_CHAT_MODEL', 'qwen3:8b'),
    temperature=0.1,  # low temperature from Lab 1 for factual answers
)
llm


ChatOllama(model='qwen3:8b', temperature=0.1)

### 1.2 Embedding model - Lab 4 / Lab 5 style


In [3]:
# Run once before using the notebook if needed:
# !ollama pull nomic-embed-text
from langchain_ollama import OllamaEmbeddings

embeddings_model = OllamaEmbeddings(model=os.getenv('OLLAMA_EMBED_MODEL', 'nomic-embed-text'))
embeddings_model


OllamaEmbeddings(model='nomic-embed-text', validate_model_on_init=False, base_url=None, client_kwargs={}, async_client_kwargs={}, sync_client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, keep_alive=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None)

## 2. Custom Dataset


In [4]:
manifest_df = pd.read_csv(DATASET_DIR / 'source_manifest.csv').fillna('')
eval_df = pd.read_csv(EVAL_FILE).fillna('')
lineage_df = pd.read_csv(LINEAGE_FILE)

print('RAG source files:', len(manifest_df))
print('Evaluation questions:', len(eval_df))
print('Lineage edges:', len(lineage_df))
display(manifest_df.head(10))


RAG source files: 41
Evaluation questions: 26
Lineage edges: 10


,source_path,title,category,pipeline,incident_id
0,docs/platform_architecture.md,Platform Architecture,docs,platform,
1,docs/lineage_overview.md,Lineage Overview,docs,platform,
2,docs/operating_principles.md,Operating Principles,docs,platform,
3,dags/orders_pipeline.py,Orders Pipeline,dags,orders,
4,dags/customers_pipeline.py,Customers Pipeline,dags,customers,
5,dags/clickstream_pipeline.py,Clickstream Pipeline,dags,clickstream,
6,dags/inventory_pipeline.py,Inventory Pipeline,dags,inventory,
7,configs/orders_source.yaml,Orders Source,configs,orders,
8,configs/airflow_connections.md,Airflow Connections,configs,platform,
9,configs/spark_sessionize.yaml,Spark Sessionize,configs,clickstream,


### Dataset design

Each source has metadata: `source_path`, `title`, `category`, `pipeline`, and `incident_id`.

The corpus covers orders, customers, clickstream, inventory and shared platform operations. The metadata lets us compare general retrieval with pipeline-scoped retrieval.


## 3. Load Documents - follows Lab 4 loader pattern


In [5]:
from langchain_community.document_loaders import TextLoader

all_docs = []
for _, row in manifest_df.iterrows():
    path = DATASET_DIR / 'knowledge_base' / row['source_path']
    docs = TextLoader(str(path), encoding='utf-8').load()
    for doc in docs:
        doc.metadata.update({
            'source': row['source_path'],
            'title': row['title'],
            'category': row['category'],
            'pipeline': row['pipeline'],
            'incident_id': row['incident_id'],
        })
    all_docs.extend(docs)

print('Loaded documents:', len(all_docs))
print(all_docs[0].metadata)


Loaded documents: 41
{'source': 'docs/platform_architecture.md', 'title': 'Platform Architecture', 'category': 'docs', 'pipeline': 'platform', 'incident_id': ''}


## 4. Split Documents - same baseline as Lab 4


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)
all_splits = text_splitter.split_documents(all_docs)
print(f'Split {len(all_docs)} documents into {len(all_splits)} chunks.')


Split 41 documents into 41 chunks.


## 5. Store Chunks in Chroma - follows Lab 4 / Lab 5


In [7]:
from langchain_chroma import Chroma

vector_store_chroma = Chroma(
    collection_name='dataflow-doctor-rag',
    embedding_function=embeddings_model,
    persist_directory=PERSIST_DIR,
)

# For a clean rerun, remove chroma_dataflow_doctor_db before adding again.
uuids = [str(uuid4()) for _ in range(len(all_splits))]
vector_store_chroma.add_documents(documents=all_splits, ids=uuids)
print('Indexed chunks:', len(all_splits))


Indexed chunks: 41


## 6. Retrieval Methods from Class


### 6.1 Similarity retrieval


In [8]:
def get_similarity_retriever(pipeline, k=4):
    kwargs = {'k': k}
    if pipeline not in {'all', 'platform'}:
        kwargs['filter'] = {'pipeline': pipeline}
    return vector_store_chroma.as_retriever(
        search_type='similarity',
        search_kwargs=kwargs,
    )


### 6.2 MMR retrieval


In [9]:
def get_mmr_retriever(pipeline, k=4):
    kwargs = {'k': k}
    if pipeline not in {'all', 'platform'}:
        kwargs['filter'] = {'pipeline': pipeline}
    return vector_store_chroma.as_retriever(
        search_type='mmr',
        search_kwargs=kwargs,
    )


### 6.3 Hybrid retrieval: BM25 + Chroma - follows Lab 4


In [10]:
from langchain.retrievers import BM25Retriever, EnsembleRetriever

def get_pipeline_splits(pipeline):
    if pipeline in {'all', 'platform'}:
        return all_splits
    selected = [d for d in all_splits if d.metadata.get('pipeline') in {pipeline, 'platform'}]
    return selected or all_splits

def get_hybrid_retriever(pipeline, k=4):
    selected_docs = get_pipeline_splits(pipeline)
    bm25_retriever = BM25Retriever.from_documents(documents=selected_docs)
    bm25_retriever.k = k

    chroma_retriever = get_similarity_retriever(pipeline, k=k)

    ensemble_retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, chroma_retriever],
        weights=[0.7, 0.3],
    )
    return ensemble_retriever


### 6.4 Inspect retrieval before generation


In [11]:
selected_pipeline = 'orders'
question = 'The orders schema validation says discount_code is unexpected. What happened?'

hybrid_retriever = get_hybrid_retriever(selected_pipeline)
retrieved_docs = hybrid_retriever.invoke(question)

for i, doc in enumerate(retrieved_docs, 1):
    print(f'--- Result {i} ---')
    print('Pipeline:', doc.metadata.get('pipeline'))
    print('Category:', doc.metadata.get('category'))
    print('Source:', doc.metadata.get('source'))
    print(doc.page_content[:500], '\n')


--- Result 1 ---
Pipeline: orders
Category: incidents
Source: incidents/INC-2026-041-orders-schema-drift.md
# INC-2026-041 Orders Schema Drift

Date: 2026-04-11
Pipeline: orders_daily_pipeline
Impact: `validate_orders_schema` failed and prevented `fct_orders` and `mart_daily_revenue` publication.

Root cause: ecommerce producer deployed schema v3 adding nullable `discount_code`, while the pipeline still validated against v2. No rows were lost because loading stopped before raw ingestion.

Resolution: approve v3 contract, update manifest, rerun from validation using the same extract. 

--- Result 2 ---
Pipeline: orders
Category: logs
Source: logs/orders_schema_failure.log
2026-04-11T02:08:14Z validate_orders_schema ERROR expected columns=6 actual columns=7
2026-04-11T02:08:14Z ERROR unexpected column: discount_code
2026-04-11T02:08:15Z DAG orders_daily_pipeline task validate_orders_schema FAILED 

--- Result 3 ---
Pipeline: platform
Category: runbooks
Source: runbooks/schema_drift.md
# 

## 7. Prompt - follows Lab 4 ChatPromptTemplate


In [12]:
from langchain_core.prompts import ChatPromptTemplate

system_message_template = """You are DataFlow Doctor, a data-pipeline and ETL debugging assistant.
Use ONLY the retrieved context supplied to you.
Do not invent schemas, row counts, SQL behavior, configs, incidents or remediation.
Separate observed evidence from hypotheses.
Prefer reversible checks before reruns or backfills.
If the context is insufficient, clearly say so.

Selected pipeline: {pipeline}

Retrieved context:
{context}
"""

dataflow_prompt = ChatPromptTemplate([
    ('system', system_message_template),
    ('human', 'Pipeline question: {question}'),
])


## 8. Structured Output - follows Lab 3


In [13]:
from pydantic import BaseModel, Field

class DataFlowDoctorResponse(BaseModel):
    issue_summary: str = Field(description='Short evidence-grounded summary')
    likely_root_causes: list[str] = Field(description='Root causes supported by retrieved context')
    affected_assets: list[str] = Field(description='Tables, jobs, pipelines or datasets implicated')
    evidence: list[str] = Field(description='Concrete observations from retrieved evidence')
    recommended_steps: list[str] = Field(description='Safe ordered next steps supported by context')
    source_documents: list[str] = Field(description='Retrieved source titles or paths used')
    data_quality_risk: str = Field(description='Possible data-quality or downstream impact')
    information_gap: bool = Field(description='True if retrieved context is insufficient')

structured_llm = llm.with_structured_output(DataFlowDoctorResponse)


## 9. LCEL RAG Chain - same idea as Lab 4


In [14]:
def format_docs(docs):
    formatted = []
    for doc in docs:
        formatted.append(
            f"SOURCE: {doc.metadata.get('source')}\n"
            f"TITLE: {doc.metadata.get('title')}\n"
            f"PIPELINE: {doc.metadata.get('pipeline')}\n"
            f"CATEGORY: {doc.metadata.get('category')}\n"
            f"CONTENT: {doc.page_content}"
        )
    return '\n\n'.join(formatted)

def ask_dataflow_lcel(pipeline, question):
    retriever = get_hybrid_retriever(pipeline)
    docs = retriever.invoke(question)
    context = format_docs(docs)
    chain = dataflow_prompt | structured_llm
    answer = chain.invoke({
        'pipeline': pipeline,
        'question': question,
        'context': context,
    })
    return answer, docs


In [15]:
# Lab 2/3 demo: LCEL chain with structured output.
answer, evidence = ask_dataflow_lcel(
    'orders',
    'The orders schema validation says discount_code is unexpected. What happened?'
)
print(answer.model_dump())


{'issue_summary': "Schema validation failed due to unexpected column 'discount_code' in the orders pipeline.", 'likely_root_causes': ["The ecommerce producer deployed schema version v3, which includes a nullable 'discount_code' column.", "The pipeline's validation step was still configured to expect schema version v2, which does not include 'discount_code'.", 'The schema version in the config (v3) was not properly synchronized with the validation logic or manifest.'], 'affected_assets': ['validate_orders_schema validation step', 'fct_orders fact table', 'mart_daily_revenue mart'], 'evidence': ["Log shows 'unexpected column: discount_code' error during schema validation.", 'Incident report confirms schema drift between v2 and v3 with new nullable column.', 'Config file shows expected_schema_version: v3, but validation failed as if expecting v2.'], 'recommended_steps': ["Verify the current schema version in the validation manifest matches the producer's deployed schema (v3).", 'Confirm t

## 10. Advanced RAG Experiment - MultiQuery from Lab 5


In [16]:
from langchain.retrievers.multi_query import MultiQueryRetriever

base_retriever = get_similarity_retriever('clickstream', k=2)
mq_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm,
    include_original=True,
)

# Example:
# mq_docs = mq_retriever.invoke('Why did the Spark job run out of memory?')
# [(d.metadata.get('source'), d.metadata.get('category')) for d in mq_docs]


## 11. Final Workflow with LangGraph - follows Lab 6


In [17]:
from typing_extensions import TypedDict
from typing import List
from langchain_core.documents import Document
from langgraph.graph import START, StateGraph, END

class State(TypedDict):
    pipeline: str
    question: str
    context: List[Document]
    answer: DataFlowDoctorResponse

def retrieve(state: State):
    retriever = get_hybrid_retriever(state['pipeline'])
    retrieved_docs = retriever.invoke(state['question'])
    return {'context': retrieved_docs}

def generate(state: State):
    context = format_docs(state['context'])
    response = (dataflow_prompt | structured_llm).invoke({
        'pipeline': state['pipeline'],
        'question': state['question'],
        'context': context,
    })
    return {'answer': response}

graph_builder = StateGraph(State)
graph_builder.add_node('retrieve', retrieve)
graph_builder.add_node('generate', generate)
graph_builder.add_edge(START, 'retrieve')
graph_builder.add_edge('retrieve', 'generate')
graph_builder.add_edge('generate', END)
graph = graph_builder.compile()


C:\Users\Eldoria\Pictures\portfolio projects\DataFlow_Doctor_End_to_End_Sir_Code\DataFlow_Doctor_End_to_End_Sir_Code\.venv\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [18]:
# Lab 6 demo: the same question answered through the LangGraph pipeline.
result = graph.invoke({
    'pipeline': 'inventory',
    'question': 'Why did inventory parsing see only one column?'
})
print(result['answer'].model_dump())


{'issue_summary': 'Inventory parsing failed due to delimiter mismatch', 'likely_root_causes': ['Vendor delivered semicolon-delimited file instead of comma-delimited'], 'affected_assets': ['inventory_snapshot'], 'evidence': ['parse_csv ERROR expected_columns=4 observed_columns=1', 'sample_line="SKU104;WH2;27;2026-08-02T04:55:00Z"'], 'recommended_steps': ['Verify file delimiter matches pipeline configuration', 'Validate vendor-delivered files against contract specifications'], 'source_documents': ['incidents/INC-2026-097-inventory-delimiter.md', 'logs/inventory_parse.log'], 'data_quality_risk': 'Single-column records would cause downstream processing failures and data loss.', 'information_gap': False}


## 12. Project-Specific Feature: Lineage Impact Analysis


In [19]:
from collections import deque

def downstream_assets(asset):
    edges = {}
    for _, row in lineage_df.iterrows():
        edges.setdefault(row['upstream_asset'], []).append((row['downstream_asset'], row['job']))
    seen = {asset}
    q = deque([(asset, 0)])
    result = []
    while q:
        current, depth = q.popleft()
        for downstream, job in edges.get(current, []):
            if downstream in seen:
                continue
            seen.add(downstream)
            result.append({'asset': downstream, 'via_job': job, 'depth': depth + 1})
            q.append((downstream, depth + 1))
    return result

display(pd.DataFrame(downstream_assets('raw.orders')))


,asset,via_job,depth
0,analytics.stg_orders,orders_transform,1
1,analytics.fct_orders,orders_transform,2
2,analytics.mart_daily_revenue,daily_revenue_mart,3
3,analytics.customer_360,customer_360,3
4,analytics.funnel_mart,funnel_mart,3


## 13. Retrieval Evaluation - Similarity vs MMR vs Hybrid


In [20]:
def category_hit(docs, expected_category):
    categories = [d.metadata.get('category') for d in docs]
    return int(expected_category in categories)

def source_hit(docs, expected_source):
    sources = [d.metadata.get('source') for d in docs]
    return int(expected_source in sources)

def reciprocal_rank(docs, expected_source):
    for rank, doc in enumerate(docs, 1):
        if doc.metadata.get('source') == expected_source:
            return 1.0 / rank
    return 0.0

def evaluate_retriever(method='hybrid', max_questions=None):
    rows = []
    subset = eval_df if max_questions is None else eval_df.head(max_questions)
    for _, row in subset.iterrows():
        pipeline = row['pipeline']
        question = row['question']
        if method == 'similarity':
            retriever = get_similarity_retriever(pipeline)
        elif method == 'mmr':
            retriever = get_mmr_retriever(pipeline)
        else:
            retriever = get_hybrid_retriever(pipeline)
        docs = retriever.invoke(question)
        rows.append({
            'question_id': row['question_id'],
            'method': method,
            'source_hit': source_hit(docs, row['expected_source']),
            'category_hit': category_hit(docs, row['expected_category']),
            'reciprocal_rank': reciprocal_rank(docs, row['expected_source']),
            'retrieved_sources': ' | '.join(d.metadata.get('source', '') for d in docs),
        })
    return pd.DataFrame(rows)


In [21]:
# Retrieval comparison over the labelled evaluation set.
similarity_results = evaluate_retriever('similarity')
mmr_results = evaluate_retriever('mmr')
hybrid_results = evaluate_retriever('hybrid')
all_results = pd.concat([similarity_results, mmr_results, hybrid_results], ignore_index=True)
display(all_results.groupby('method')[['source_hit','category_hit','reciprocal_rank']].mean().round(3))


,source_hit,category_hit,reciprocal_rank
method,,,
hybrid,0.808,0.885,0.573
mmr,0.692,0.731,0.535
similarity,0.654,0.654,0.481


## 14. Controlled Failure Test


In [22]:
# Guardrail check: information_gap should be true for questions outside the corpus.
unsupported = graph.invoke({
    'pipeline': 'orders',
    'question': "Who won yesterday's cricket match?"
})
print(unsupported['answer'].model_dump())


{'issue_summary': "The question 'Who won yesterday's cricket match?' is unrelated to the 'orders' pipeline context provided. The available documents only cover data pipeline transformations, revenue reconciliation, and order data quality checks. There is no information about cricket matches or sports results in the context.", 'likely_root_causes': ["The question pertains to a sports event, which is outside the scope of the 'orders' pipeline and its associated documentation."], 'affected_assets': [], 'evidence': ['All retrieved context documents (transformations, data quality, logs) are focused on order processing, revenue calculations, and data pipeline operations.'], 'recommended_steps': ["Consult sports news sources or official cricket match reports for information on yesterday's match outcome."], 'source_documents': ['transformations/mart_daily_revenue.sql', 'data_quality/revenue_reconciliation.md', 'data_quality/orders_expectations.md', 'logs/warehouse_pool.log', 'logs/orders_dupli

## 15. Limitations

1. The bundled corpus is synthetic and designed for an academic/portfolio demonstration.
2. Production pipeline debugging would require secure access to live logs, orchestration metadata and warehouse observability.
3. Lineage in this project is intentionally small and explicit.
4. Retrieval quality depends on corpus quality, chunking and embedding/retrieval choices.
5. The assistant provides decision support and does not automatically rerun jobs or modify production data.


## 16. AI Usage Disclosure

AI tools were used as a helper for project framing, documentation, code review and debugging suggestions. The core implementation intentionally follows the syntax and patterns taught in the six in-class notebooks. Kushagra should run, understand and be able to explain the final notebook, dataset design, retrieval methods, evaluation and demo flow.
